# 📊 EDA – Phân tích dữ liệu bán hàng đa kênh
**Mục tiêu:** Khám phá dữ liệu từ Data Warehouse, phát hiện trend, seasonality và outliers trước khi train mô hình Prophet.

**Kết nối:** PostgreSQL Data Warehouse (sales_analytics_ai_db / schema dw)

In [ ]:
# Cài đặt thư viện (chạy lần đầu trên Colab)
!pip install psycopg2-binary pandas matplotlib seaborn plotly -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import psycopg2
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12
print('✅ Import thành công')

In [ ]:
# ── Cấu hình kết nối Database ──────────────────────────────────────────────
# Thay thế bằng thông tin thực tế hoặc dùng Google Colab Secrets
DB_CONFIG = {
    'host':     'localhost',      # hoặc IP server của bạn
    'port':     5432,
    'database': 'sales_analytics_ai_db',
    'user':     'postgres',
    'password': 'your_password',  # thay bằng password thực tế
}

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

def query(sql, params=None):
    with get_conn() as conn:
        return pd.read_sql(sql, conn, params=params)

# Test kết nối
try:
    df_test = query('SELECT 1 AS ok')
    print('✅ Kết nối Database thành công')
except Exception as e:
    print(f'❌ Lỗi kết nối: {e}')
    print('💡 Nếu dùng Colab, hãy dùng ngrok hoặc Cloud SQL Proxy để tunnel DB')

## 1. Tổng quan dữ liệu

In [ ]:
# Tải dữ liệu tổng hợp theo ngày từ fact_sales + dim_date
SQL_DAILY = """
    SELECT
        dd.full_date          AS date,
        dc.channel_name       AS channel,
        SUM(fs.net_revenue)   AS net_revenue,
        SUM(fs.gross_revenue) AS gross_revenue,
        SUM(fs.order_count)   AS order_count,
        SUM(fs.item_quantity) AS item_quantity,
        SUM(fs.gross_profit)  AS gross_profit,
        SUM(fs.discount_amount) AS discount_amount
    FROM dw.fact_sales fs
    JOIN dw.dim_date    dd ON fs.date_id    = dd.date_id
    JOIN dw.dim_channel dc ON fs.channel_id = dc.channel_id
    WHERE fs.order_status = 'DELIVERED'
    GROUP BY dd.full_date, dc.channel_name
    ORDER BY dd.full_date
"""
df_daily = query(SQL_DAILY)
df_daily['date'] = pd.to_datetime(df_daily['date'])

print(f'📦 Tổng số ngày có dữ liệu: {df_daily["date"].nunique()}')
print(f'📦 Kênh bán hàng: {df_daily["channel"].unique()}')
print(f'📦 Khoảng thời gian: {df_daily["date"].min().date()} → {df_daily["date"].max().date()}')
df_daily.head()

In [ ]:
# Thống kê mô tả
df_total = df_daily.groupby('date').agg(
    net_revenue=('net_revenue', 'sum'),
    order_count=('order_count', 'sum'),
    gross_profit=('gross_profit', 'sum'),
).reset_index()

print('=== Thống kê tổng hợp ===')
print(df_total[['net_revenue','order_count','gross_profit']].describe().round(0))

## 2. Phân tích Trend doanh thu

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Doanh thu theo ngày (tất cả kênh)
ax1 = axes[0]
ax1.plot(df_total['date'], df_total['net_revenue'], color='steelblue', linewidth=1.5, label='Doanh thu thuần')
ax1.fill_between(df_total['date'], df_total['net_revenue'], alpha=0.15, color='steelblue')
# Đường trend 7 ngày
df_total['ma7']  = df_total['net_revenue'].rolling(7).mean()
df_total['ma30'] = df_total['net_revenue'].rolling(30).mean()
ax1.plot(df_total['date'], df_total['ma7'],  color='orange',  linewidth=2, label='MA 7 ngày')
ax1.plot(df_total['date'], df_total['ma30'], color='red',     linewidth=2, linestyle='--', label='MA 30 ngày')
ax1.set_title('Doanh thu thuần theo ngày – Tất cả kênh', fontsize=14, fontweight='bold')
ax1.set_ylabel('Doanh thu (VNĐ)')
ax1.legend()
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
ax1.tick_params(axis='x', rotation=45)
ax1.grid(alpha=0.3)

# Số đơn hàng
ax2 = axes[1]
ax2.bar(df_total['date'], df_total['order_count'], color='seagreen', alpha=0.7, label='Số đơn hàng')
ax2.set_title('Số đơn hàng theo ngày', fontsize=14, fontweight='bold')
ax2.set_ylabel('Số đơn')
ax2.legend()
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
ax2.tick_params(axis='x', rotation=45)
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('eda_trend_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu eda_trend_revenue.png')

## 3. Phân tích Seasonality

In [ ]:
df_total['weekday']      = df_total['date'].dt.day_name()
df_total['weekday_num']  = df_total['date'].dt.dayofweek
df_total['month']        = df_total['date'].dt.month
df_total['month_name']   = df_total['date'].dt.strftime('%b')
df_total['week_of_year'] = df_total['date'].dt.isocalendar().week

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Doanh thu trung bình theo thứ trong tuần
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday_vi    = ['T.2','T.3','T.4','T.5','T.6','T.7','CN']
df_by_weekday = df_total.groupby('weekday_num')['net_revenue'].mean().reset_index()
axes[0].bar(weekday_vi, df_by_weekday['net_revenue'], color='cornflowerblue', edgecolor='white')
axes[0].set_title('Doanh thu TB theo thứ trong tuần', fontweight='bold')
axes[0].set_ylabel('Doanh thu TB (VNĐ)')
axes[0].grid(alpha=0.3, axis='y')

# Doanh thu trung bình theo tháng
df_by_month = df_total.groupby('month')['net_revenue'].mean().reset_index()
axes[1].bar(df_by_month['month'], df_by_month['net_revenue'], color='salmon', edgecolor='white')
axes[1].set_title('Doanh thu TB theo tháng', fontweight='bold')
axes[1].set_xlabel('Tháng')
axes[1].set_xticks(range(1, 13))
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('eda_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Phân tích theo kênh bán hàng

In [ ]:
df_channel = df_daily.groupby('channel').agg(
    net_revenue=('net_revenue','sum'),
    order_count=('order_count','sum'),
    gross_profit=('gross_profit','sum'),
).reset_index()
df_channel['profit_margin'] = df_channel['gross_profit'] / df_channel['net_revenue'] * 100
df_channel['revenue_pct']   = df_channel['net_revenue'] / df_channel['net_revenue'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart tỷ trọng doanh thu
axes[0].pie(
    df_channel['net_revenue'],
    labels=df_channel['channel'],
    autopct='%1.1f%%',
    startangle=90,
    colors=['#4C72B0','#DD8452','#55A868','#C44E52'],
)
axes[0].set_title('Tỷ trọng doanh thu theo kênh', fontweight='bold')

# Bar chart profit margin
axes[1].barh(df_channel['channel'], df_channel['profit_margin'], color='mediumseagreen')
axes[1].set_title('Tỷ suất lợi nhuận (%) theo kênh', fontweight='bold')
axes[1].set_xlabel('Profit Margin (%)')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('eda_channel_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(df_channel.to_string(index=False))

## 5. Phát hiện Outliers

In [ ]:
# Phát hiện outlier bằng IQR
Q1 = df_total['net_revenue'].quantile(0.25)
Q3 = df_total['net_revenue'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_total['is_outlier'] = (
    (df_total['net_revenue'] < lower_bound) |
    (df_total['net_revenue'] > upper_bound)
)
df_outliers = df_total[df_total['is_outlier']]

plt.figure(figsize=(16, 5))
plt.plot(df_total['date'], df_total['net_revenue'], color='steelblue', linewidth=1, alpha=0.8)
plt.scatter(df_outliers['date'], df_outliers['net_revenue'],
            color='red', s=60, zorder=5, label=f'Outlier ({len(df_outliers)} ngày)')
plt.axhline(upper_bound, color='orange', linestyle='--', label=f'Upper bound = {upper_bound:,.0f}')
plt.axhline(lower_bound, color='orange', linestyle=':', label=f'Lower bound = {lower_bound:,.0f}')
plt.title('Phát hiện Outliers – Doanh thu theo ngày (IQR method)', fontweight='bold')
plt.ylabel('Doanh thu (VNĐ)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('eda_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📌 Số ngày outlier: {len(df_outliers)}')
print(df_outliers[['date','net_revenue']].to_string(index=False))

## 6. Chuẩn bị dataset cho Prophet

In [ ]:
# Prophet yêu cầu DataFrame có đúng 2 cột: ds (datetime) và y (giá trị)
df_prophet = df_total[['date', 'net_revenue']].rename(
    columns={'date': 'ds', 'net_revenue': 'y'}
).dropna()

# Loại bỏ outlier cực đoan (> 3 sigma) để tránh model bị lệch
mean_y, std_y = df_prophet['y'].mean(), df_prophet['y'].std()
df_prophet = df_prophet[
    (df_prophet['y'] >= mean_y - 3*std_y) &
    (df_prophet['y'] <= mean_y + 3*std_y)
].reset_index(drop=True)

print(f'✅ Dataset cho Prophet: {len(df_prophet)} ngày')
print(f'   Từ: {df_prophet["ds"].min().date()} → {df_prophet["ds"].max().date()}')
print(f'   y trung bình: {df_prophet["y"].mean():,.0f} VNĐ/ngày')

# Lưu dataset
df_prophet.to_csv('prophet_dataset.csv', index=False)
print('\n✅ Đã lưu prophet_dataset.csv → dùng cho notebook 02_Prophet_Training')